# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id's
print('Available Record Sets:')
for rs in dataset.record_sets:
    print(f"  - @id: {rs.id!r} | name: {rs.name}")

# For each record set, list all available fields and their @id's
for rs in dataset.record_sets:
    print(f"\nFields for record set @id: {rs.id!r} ({rs.name})")
    for field in rs.fields:
        print(f"    - @id: {field.id!r:40} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record set @id's into a list
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set @id's:", record_set_ids)
dataframes = {}

for rs_id in record_set_ids:
    records_iter = dataset.records(record_set=rs_id)
    df = pd.DataFrame(records_iter)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set: {rs_id}")

# Pick the first record set (if only one; otherwise adjust for your needs)
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print("Available columns in DataFrame:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print('No record sets found!')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section prepares the data for further analysis and visualization.

In [ ]:
# Pick the record set to work on
record_set_id = first_rs_id
df = dataframes[record_set_id]

# Show which fields/columns are numeric
print("Numeric fields in the chosen record set:")
print(df.select_dtypes(include=[np.number]).columns.tolist())

# If at least one numeric field is found, pick the first for demonstration
numeric_columns = df.select_dtypes(include=[np.number]).columns
if len(numeric_columns) > 0:
    numeric_field = numeric_columns[0]
    print(f"Using numeric field for filtering and normalization: {numeric_field}")
    threshold = df[numeric_field].mean()  # Use mean as a threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (choose the first object-type/column if available)
    possible_groups = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    if possible_groups:
        group_field = possible_groups[0]
        print(f"Grouping on field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean of {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group-by field (categorical) found in dataset.")
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

In [ ]:
# Simple visualization: Histogram of the numeric field (if available)
if len(df.select_dtypes(include=[np.number]).columns) > 0:
    field = numeric_columns[0]
    plt.figure(figsize=(8,5))
    df[field].hist(bins=15, color='skyblue', edgecolor='black')
    plt.title(f'Histogram of {field}')
    plt.xlabel(field)
    plt.ylabel('Count')
    plt.show()
    
    # If there is a categorical field, show boxplot
    if possible_groups:
        group_field = possible_groups[0]
        df[[group_field, field]].boxplot(by=group_field, column=field, figsize=(10,6))
        plt.title(f'Boxplot of "{field}" grouped by "{group_field}"')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(field)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we loaded a FAIR^2 compliant clinicopathological colorectal cancer dataset and explored its structure with `mlcroissant`. We reviewed record sets and fields using their `@id`s, extracted records into DataFrames, conducted EDA such as filtering and normalization, and visualized key numeric field distributions. With the schema-driven approach, you can confidently refer to fields and structures by stable `@id` regardless of column renaming, aiding reproducible research and machine learning pipeline development.